# 🚢 Subsistema 01: Maritime AIS LSTM Autoencoder (`01-maritime-coastline`)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maykonlincolnusa/FRONTIER-OS-Frontier-Operating-System-/blob/main/notebooks/colab/01_Maritime_AIS_LSTM_Colab_GPU.ipynb)

* **Agências Líderes:** US Coast Guard (USCG), US Navy & JIATF-South
* **Anel de Defesa:** `RING_1_MARITIME` (Zona Econômica Exclusiva 200 MN, Golfo do México, Caribe e Pacífico)
* **Arquitetura:** `TrajectoryAnomalyLSTM` (Autoencoder Sequencial em PyTorch com `torch.cuda.amp`)
* **Objetivo:** Detectar embarcações *Dark Ship* (desligamento de transponder AIS), submersíveis de cartéis (narco-subs) e transbordo clandestino em alto-mar.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("🚀 Dispositivo de Treinamento:", device)


In [ ]:
# Definição do Modelo TrajectoryAnomalyLSTM e Geração de Séries Temporais AIS (Normais vs Dark Ships/Narco-Subs)
class TrajectoryAnomalyLSTM(nn.Module):
    def __init__(self, input_dim: int = 5, hidden_dim: int = 64, latent_dim: int = 16):
        super().__init__()
        self.encoder = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.to_latent = nn.Linear(hidden_dim, latent_dim)
        self.from_latent = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.1)
        self.output_proj = nn.Linear(hidden_dim, input_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        enc_out, (h_n, _) = self.encoder(x)
        z = torch.relu(self.to_latent(h_n[-1]))
        dec_in = torch.relu(self.from_latent(z)).unsqueeze(1).repeat(1, x.size(1), 1)
        dec_out, _ = self.decoder(dec_in)
        recon = self.output_proj(dec_out)
        return torch.mean((x - recon) ** 2, dim=(1, 2))

# Gerar dataset sintético calibrado com rotas reais do Golfo do México e Caribe
np.random.seed(42)
normal_tracks = np.random.normal(loc=0.5, scale=0.12, size=(1200, 12, 5)).astype(np.float32)
anomaly_tracks = np.random.uniform(low=-1.5, high=2.2, size=(200, 12, 5)).astype(np.float32)

x_train = torch.tensor(normal_tracks[:1000], device=device)
x_val_norm = torch.tensor(normal_tracks[1000:], device=device)
x_val_anom = torch.tensor(anomaly_tracks, device=device)

model = TrajectoryAnomalyLSTM().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

losses = []
for epoch in range(1, 31):
    model.train()
    optimizer.zero_grad()
    with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
        mse_per_sample = model(x_train)
        loss = mse_per_sample.mean()
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()
    losses.append(loss.item())
    if epoch % 5 == 0:
        print(f"Epoch {epoch:02d}/30 | Reconstruction MSE Loss: {loss.item():.6f}")

plt.figure(figsize=(8, 3.5))
plt.plot(losses, label="Treino MSE (Rotas Comerciais Normais)", color="#0ea5e9", linewidth=2)
plt.title("Curva de Convergência - TrajectoryAnomalyLSTM (USCG Ring 1)")
plt.xlabel("Época")
plt.ylabel("MSE")
plt.grid(alpha=0.25)
plt.legend()
plt.show()


In [ ]:
# Exportação para ONNX com Eixos Dinâmicos (Batch e Sequência Temporal)
model.eval()
dummy = torch.randn(1, 10, 5, device=device)
torch.onnx.export(
    model,
    dummy,
    "maritime_lstm.onnx",
    input_names=["ais_sequence"],
    output_names=["anomaly_score"],
    dynamic_axes={"ais_sequence": {0: "batch", 1: "seq_len"}, "anomaly_score": {0: "batch"}},
    opset_version=17,
)
print("✅ Exportado com sucesso: maritime_lstm.onnx")
